# PRODUCT AND SALES ANALYTICS 
**OBJECTIVE**
The objective of this notebook is to analyze product performance across the Olist marketplace. 
This notebook will answer questions like:
1. Which products generate the highest revenue ? 
2. Which porduct categories are most popular ? 
3. Which categories are growing ? 
4. Which categories are most expensive ? 
5. which categories incur the highest frieght cost ? 
6. which categories should the business invest in ? 

In [1]:
import warnings 
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns 

In [2]:
# we don't need all the csv anymore, so we load only relevant data. 
project_root = Path.cwd().parent.parent
data_dir = project_root / "datasets" / "Olist"


In [3]:
orders = pd.read_csv(data_dir/"olist_orders_dataset.csv",
                     parse_dates=["order_purchase_timestamp", 
                                  "order_approved_at", 
                                  "order_delivered_carrier_date", 
                                  "order_delivered_customer_date", 
                                  "order_estimated_delivery_date"])
order_items = pd.read_csv(data_dir/"olist_order_items_dataset.csv")

products = pd.read_csv(data_dir/"olist_products_dataset.csv")

translation = pd.read_csv(data_dir/"product_category_name_translation.csv")


In [4]:
# merging product tables 
products_df = (
    order_items.merge(products, on="product_id", how="left").merge(translation, on="product_category_name", how="left")
)
products_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools


In [6]:
# top categories by revenue 
category_revenue = (
    products_df
    .groupby("product_category_name_english")
    .agg(
        Revenue=("price", "sum"),
        Orders=("order_id", "count")
    )
    .sort_values("Revenue", ascending=False)
    .reset_index()
)

In [7]:
category_revenue.head(10)

,product_category_name_english,Revenue,Orders
0,health_beauty,1258681.34,9670
1,watches_gifts,1205005.68,5991
2,bed_bath_table,1036988.68,11115
3,sports_leisure,988048.97,8641
4,computers_accessories,911954.32,7827
5,furniture_decor,729762.49,8334
6,cool_stuff,635290.85,3796
7,housewares,632248.66,6964
8,auto,592720.11,4235
9,garden_tools,485256.46,4347


In [8]:
fig=px.bar(
    category_revenue.head(10),
    x="Revenue",
    y="product_category_name_english",
    orientation="h",
    title="Top 10 Product Categories by Revenue",
    template="plotly_white",
)
fig.show()

In [10]:
# top selling products 
top_products = (
    products_df
    .groupby("product_id")
    .agg(
        Orders =("order_id", "count"),
        Revenue = ("price", "sum")
    )
    .sort_values("Orders",ascending=False)
    .head(20)
    .reset_index()
)
top_products.head(10)

,product_id,Orders,Revenue
0,aca2eb7d00ea1a7b8ebd4e68314663af,527,37608.90
1,99a4788cb24856965c36a24e339b6058,488,43025.56
2,422879e10f46682990de24d770e7f83d,484,26577.22
3,389d119b48cf3043d311335e499d9c6b,392,21440.59
4,368c6c730842d78016ad823897a372db,388,21056.80
5,53759a2ecddad2bb87a079a1f1519f73,373,20387.20
6,d1c427060a0f73f6b889a5c7c61f2ac4,343,47214.51
7,53b36df67ebb7c41585e8d54d6772e08,323,37683.42
8,154e7e31ebfa092203795c972e5804a6,281,6325.19
9,3dd2a17168ec895c781a9191c1e95ad7,274,41082.60


In [12]:
fig = px.bar(
    top_products,
    x="product_id",
    y="Orders",
    title="top selling products",
    template="plotly_white"
)
fig.update_xaxes(showticklabels=False)
fig.show()

In [ ]:
# highest average product price 
category_price=(
    products_df
    .groupby("product_category_name_english")
    .agg(
        Average_Price = ("price","mean")    
    )
    .sort_values("Average_Price",ascending=False).head(10)
)
category_price.head(10)

,Average_Price
product_category_name_english,
computers,1098.340542
small_appliances_home_oven_and_coffee,624.285658
home_appliances_2,476.124958
agro_industry_and_commerce,342.124858
musical_instruments,281.616000
small_appliances,280.778468
fixed_telephony,225.693182
construction_tools_safety,208.992371
watches_gifts,201.135984


In [18]:
category_price = category_price.reset_index()
fig = px.bar(
    category_price,
    x="Average_Price",
    y="product_category_name_english",
    orientation="h",
    title="Highest Average Product Prices by Category",
    template="plotly_white"
)
fig.update_xaxes(showticklabels=False)
fig.show()

In [19]:
# freight cost analysis
freight = (
    products_df
    .groupby("product_category_name_english")
    .agg(
        AverageFreight=("freight_value","mean")
    )
    .sort_values(
        "AverageFreight",
        ascending=False
    )
    .head(10)
)
freight=freight.reset_index()
freight

,product_category_name_english,AverageFreight
0,computers,48.454680
1,home_appliances_2,44.538571
2,furniture_mattress_and_upholstery,42.906842
3,kitchen_dining_laundry_garden_furniture,42.702598
4,furniture_bedroom,42.497523
5,office_furniture,40.551124
6,small_appliances_home_oven_and_coffee,36.156053
7,furniture_living_room,35.722008
8,signaling_and_security,32.702613
9,industry_commerce_and_business,29.420448


In [21]:
fig=px.bar(
    freight,
    x="AverageFreight",
    y="product_category_name_english",
    orientation="h",
    title="Highest Freight Cost Categories",
    template="plotly_white"
)
fig.show()

In [22]:
# Monthly category sales 
sales =(
    orders[["order_id", "order_purchase_timestamp"]]
    .merge(order_items, on="order_id")
    .merge(products, on="product_id")
    .merge(translation, on="product_category_name",
    how="left")
)
sales["Month"]=(
    sales["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)


In [23]:
top5=(
    sales["product_category_name_english"]
    .value_counts()
    .head(5)
    .index
)
monthly=(
    sales[sales["product_category_name_english"].isin(top5)
          ]
        .groupby(
            [
                "Month","product_category_name_english"
            ]
        )["price"]
        .sum()
        .reset_index()
)

In [24]:
fig = px.line(
    monthly,
    x="Month",
    y="price",
    color="product_category_name_english",
    markers=True,
    title="Monthly Revenue by top categories",
    template="plotly_white"
)
fig.show()

# Product & Sales Insights

The product analytics reveal several important business trends across the Olist marketplace.

### Revenue Concentration

- Revenue is concentrated in a relatively small number of product categories.
- The **Health & Beauty** category is the highest revenue generator, followed by **Watches & Gifts**, **Bed Bath & Table**, **Sports & Leisure**, and **Computers & Accessories**.
- These categories should remain strategic priorities for inventory planning and promotional campaigns.

### Sales Volume

- The **Bed Bath & Table** category records the highest number of orders, demonstrating consistently strong customer demand.
- High order volume does not always correspond to the highest revenue, indicating differences in average selling prices across categories.

### Premium Product Categories

- Categories such as **Computers**, **Small Home Appliances**, and **Home Appliances** have the highest average selling prices.
- These products generate greater revenue per transaction despite comparatively lower sales volumes.
- Premium categories can significantly improve profitability through targeted marketing and premium customer experiences.

### Freight Cost Analysis

- High-value and bulky product categories generally incur the highest freight costs.
- Categories such as **Computers** and **Home Appliances** have significantly higher average shipping costs than most other categories.
- Optimizing shipping costs for these categories could improve overall profit margins.

### Category Trends

- Different product categories exhibit varying sales patterns over time.
- Seasonal fluctuations suggest that customer demand changes throughout the year.
- These trends can support inventory planning, promotional scheduling, and demand forecasting.